<a href="https://colab.research.google.com/github/erokemwa/Blog-AI/blob/master/chat_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sqlite3
from datetime import datetime
import time
import contextlib
from typing import Optional, Tuple
import signal
import sys

# ---------- Configuration ----------
DATABASE_NAME = "interactions.db"
DEFAULT_USER = "user_123"
RESPONSE_DELAY = (0.3, 0.7)  # Random delay range in seconds

# ---------- Database Schema ----------
SCHEMA = """
CREATE TABLE IF NOT EXISTS interactions(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    input TEXT NOT NULL,
    output TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    feedback INTEGER CHECK(feedback IN (0, 1))
)
"""

# ---------- Mock LLM Service ----------
class MockLLM:
    @staticmethod
    def generate_response(user_input: str) -> str:
        """Simulate LLM response with random delay and basic NLP"""
        time.sleep(random.uniform(*RESPONSE_DELAY))

        user_input = user_input.lower().strip()

        responses = {
            "hello": "Hello! How can I assist you today?",
            "bye": "Goodbye! Have a wonderful day!",
            "help": "I can help with general inquiries. Feel free to ask me anything!",
            "default": f"I understand you're asking about: {user_input.capitalize()}. Can you please elaborate?"
        }

        return responses.get(
            next((k for k in responses if user_input.startswith(k)), 'default'),
            responses['default']
        )

# ---------- Database Manager ----------
class DatabaseManager:
    def __init__(self, db_name: str = DATABASE_NAME):
        self.db_name = db_name
        self._initialize_db()

    def _initialize_db(self):
        """Initialize database with proper schema and indexes"""
        with self._get_connection(commit=True) as conn:
            conn.execute(SCHEMA)
            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_user_interactions
                ON interactions(user_id, timestamp)
            """)

    @contextlib.contextmanager
    def _get_connection(self, commit: bool = False):
        """Context manager for safe database connections"""
        conn = sqlite3.connect(self.db_name)
        try:
            yield conn
            if commit:
                conn.commit()
        except Exception as e:
            conn.rollback()
            raise e
        finally:
            conn.close()

    def save_interaction(self, user_id: str, user_input: str, llm_output: str) -> int:
        """Save interaction and return inserted row ID"""
        with self._get_connection(commit=True) as conn:
            cursor = conn.execute(
                """INSERT INTO interactions
                (user_id, input, output, timestamp)
                VALUES (?, ?, ?, ?)""",
                (user_id, user_input, llm_output, datetime.utcnow().isoformat())
            )
            return cursor.lastrowid

    def add_feedback(self, interaction_id: int, feedback: int):
        """Update interaction with user feedback"""
        with self._get_connection(commit=True) as conn:
            conn.execute(
                "UPDATE interactions SET feedback = ? WHERE id = ?",
                (feedback, interaction_id)
            )

# ---------- Main Application ----------
class ChatApp:
    def __init__(self):
        self.llm = MockLLM()
        self.db = DatabaseManager()
        self.current_user = DEFAULT_USER
        self.current_interaction: Optional[int] = None

    def process_input(self, user_input: str) -> Tuple[str, int]:
        """Process user input through LLM and store interaction"""
        response = self.llm.generate_response(user_input)
        interaction_id = self.db.save_interaction(
            self.current_user, user_input, response
        )
        self.current_interaction = interaction_id
        return response, interaction_id

    def collect_feedback(self, interaction_id: int):
        """Collect and store user feedback for an interaction"""
        while True:
            try:
                feedback = input("\nWas this helpful? (1=Yes/0=No/Enter=Skip): ")
                if not feedback:
                    return

                feedback_int = int(feedback)
                if feedback_int not in {0, 1}:
                    raise ValueError

                self.db.add_feedback(interaction_id, feedback_int)
                return
            except ValueError:
                print("Please enter 1 (Yes), 0 (No), or press Enter to skip")

# ---------- CLI Interface ----------
class ChatInterface:
    @staticmethod
    def display_welcome():
        print("\n=== AI Assistant ===")
        print("Type 'exit' to quit\n")

    @staticmethod
    def get_input() -> str:
        return input("You: ").strip()

    @staticmethod
    def show_response(response: str):
        print(f"\nAssistant: {response}")

    @staticmethod
    def show_typing_indicator():
        print("\nAssistant is typing...", end='\r')

# ---------- Signal Handling ----------
def handle_exit(signum, frame):
    print("\n\nSession ended gracefully. Goodbye!")
    sys.exit(0)

# ---------- Main Application Flow ----------
def main():
    signal.signal(signal.SIGINT, handle_exit)

    app = ChatApp()
    interface = ChatInterface()

    interface.display_welcome()

    while True:
        try:
            user_input = interface.get_input()

            if user_input.lower() in {'exit', 'quit'}:
                print("\nGoodbye!")
                break

            if not user_input:
                continue

            interface.show_typing_indicator()
            response, interaction_id = app.process_input(user_input)
            interface.show_response(response)

            app.collect_feedback(interaction_id)

        except KeyboardInterrupt:
            continue
        except Exception as e:
            print(f"\nError: {str(e)}")
            continue

if __name__ == "__main__":
    main()


=== AI Assistant ===
Type 'exit' to quit

You: hello


Assistant: Hello! How can I assist you today?

Was this helpful? (1=Yes/0=No/Enter=Skip): 1
You: bye


Assistant: Goodbye! Have a wonderful day!

Was this helpful? (1=Yes/0=No/Enter=Skip): 1


Session ended gracefully. Goodbye!


SystemExit: 0

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
